In [4]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plot
import cls_feature_class
import cls_data_generator
import parameters
import time
from time import gmtime, strftime
import torch
import torchaudio
import torch.nn as nn
import torch.optim as optim
plot.switch_backend('agg')
from IPython import embed
from cls_compute_seld_results import ComputeSELDResults, reshape_3Dto2D
from SELD_evaluation_metrics import distance_between_cartesian_coordinates
import seldnet_model 
from model import NGCCModel
from speechbrain.nnet.losses import PitWrapper 
from cst_former.CST_former_model import CST_former
from torchinfo import summary
from warmup_scheduler import GradualWarmupScheduler
import random
import pandas as pd

In [5]:
task_id = '9'
params = parameters.get_params(task_id)

SET: 9
RAW AUDIO CHUNKS w/ NGCC model + multi ACCDOA, TDOA-pretraining

	quick_test: False
	finetune_mode: False
	dataset_dir: ../../../data/data_2024/
	feat_label_dir: ../../../data/data_2024/seld_feat_label/
	model_dir: models_audio
	dcase_output_dir: results_audio
	mode: dev
	dataset: mic
	fs: 24000
	hop_len_s: 0.02
	label_hop_len_s: 0.1
	max_audio_len_s: 60
	nb_mel_bins: 64
	use_salsalite: False
	raw_chunks: True
	saved_chunks: True
	fmin_doa_salsalite: 50
	fmax_doa_salsalite: 2000
	fmax_spectra_salsalite: 9000
	model: ngccmodel
	modality: audio
	multi_accdoa: True
	thresh_unify: 15
	label_sequence_length: 1
	batch_size: 32
	eval_batch_size: 64
	dropout_rate: 0.05
	nb_cnn2d_filt: 64
	f_pool_size: [4, 4, 2]
	nb_heads: 8
	nb_self_attn_layers: 2
	nb_transformer_layers: 2
	nb_rnn_layers: 2
	rnn_size: 128
	nb_fnn_layers: 1
	fnn_size: 128
	nb_epochs: 1
	eval_freq: 25
	lr: 0.0001
	final_lr: 0
	weight_decay: 0.05
	predict_tdoa: True
	warmup: 0
	relative_dist: True
	no_dist: False
	average:

In [6]:
test_splits = [[4]]
val_splits = [[4]]
train_splits = [[3]]

In [9]:
(str(train_splits[0]))
data_gen_train = cls_data_generator.DataGenerator(
params=params, split=train_splits[0]
            )

(str(val_splits[0]))
data_gen_val= cls_data_generator.DataGenerator(
                params=params, split=val_splits[0], shuffle=False, per_file = True)
            

Computing some stats about the dataset
	Datagen_mode: dev, nb_files: 90, nb_classes:13
	nb_frames_file: 5000, feat_len: 480, nb_ch: 4, label_len:None

	Dataset: mic, split: [3]
	batch_size: 32, feat_seq_len: 5, label_seq_len: 1, shuffle: True
	Total batches in dataset: 4666
	label_dir: ../../../data/data_2024/seld_feat_label/mic_dev_adpit_label
 	feat_dir: ../../../data/data_2024/seld_feat_label/mic_dev_raw_chunks_norm

Computing some stats about the dataset
	Datagen_mode: dev, nb_files: 78, nb_classes:13
	nb_frames_file: 28462, feat_len: 480, nb_ch: 4, label_len:None

	Dataset: mic, split: [4]
	batch_size: 5693, feat_seq_len: 5, label_seq_len: 1, shuffle: False
	Total batches in dataset: 78
	label_dir: ../../../data/data_2024/seld_feat_label/mic_dev_adpit_label
 	feat_dir: ../../../data/data_2024/seld_feat_label/mic_dev_raw_chunks_norm



In [18]:
for values in data_gen_train.generate(): #batch size
    if len(values) == 2:
        data, target = values
        print(data.shape, target.shape)
        break

(32, 4, 5, 480) (32, 1, 6, 5, 13)
